[download this notebook here](https://github.com/HumanCompatibleAI/imitation/blob/master/docs/tutorials/1_train_bc.ipynb)
# Train an Agent using Behavior Cloning

Behavior cloning is the most naive approach to imitation learning. 
We take the transitions of trajectories taken by some expert and use them as training samples to train a new policy.
The method has many drawbacks and often does not work. 
However in this example, where we use an agent for the seals/CartPole-v0 environment, it is feasible.

Note that we use a variant of the CartPole environment from the seals package, which has fixed episode durations. Read more about why we do this [here](https://imitation.readthedocs.io/en/latest/main-concepts/variable_horizon.html).

First we need some kind of expert in CartPole so we can sample some expert trajectories.
For convenience we just download one from the HuggingFace model hub.

If you want to train an expert yourself have a look at the [training documenation](https://rl-baselines3-zoo.readthedocs.io/en/master/guide/train.html#basic-usage) of RL Baselines3 Zoo.

In [17]:
import numpy as np
import gymnasium as gym
from imitation.policies.serialize import load_policy
from imitation.util.util import make_vec_env
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.data.types import TrajectoryWithRew, remove_acts
from matplotlib import pyplot as plt
import matplotlib
%matplotlib widget

In [18]:
env = make_vec_env(
    "seals:seals/CartPole-v0",
    rng=np.random.default_rng(),
    post_wrappers=[
        lambda env, _: RolloutInfoWrapper(env)
    ],  # needed for computing rollouts later
)
expert = load_policy(
    "ppo-huggingface",
    organization="HumanCompatibleAI",
    env_name="seals/CartPole-v0",
    venv=env,
)

Let's quickly check if the expert is any good.
We usually should be able to reach a reward of 500, which is the maximum achievable value.

In [19]:
from stable_baselines3.common.evaluation import evaluate_policy

reward, _ = evaluate_policy(expert, env, 10)
print(reward)

500.0


Now we can use the expert to sample some trajectories.
We flatten them right away since we are only interested in the individual transitions for behavior cloning.
`imitation` comes with a number of helper functions that makes collecting those transitions really easy. First we collect 50 episode rollouts, then we flatten them to just the transitions that we need for training.

Note that the rollout function requires a vectorized environment and needs the `RolloutInfoWrapper` around each of the environments. This is why we passed the `post_wrappers` argument to `make_vec_env` above.

In [20]:
from imitation.data import rollout

rng = np.random.default_rng()
rollouts = rollout.rollout(
    expert,
    env,
    rollout.make_sample_until(min_timesteps=None, min_episodes=50),
    rng=rng,
)
transitions = rollout.flatten_trajectories(rollouts)
obs_seqs = remove_acts(rollouts)
obvstrans = rollout.flatten_observation_sequences(rollouts)

In [21]:
obvstrans[0]

{'obs': array([ 0.04689329,  0.04290264, -0.03223074,  0.01088516], dtype=float32),
 'infos': {},
 'next_obs': array([ 0.04775134, -0.1517426 , -0.03201304,  0.2932272 ], dtype=float32),
 'dones': np.False_}

In [22]:
type(rollouts[0])

imitation.data.types.TrajectoryWithRew

Let's have a quick look at what we just generated using those library functions:

In [23]:
print(
    f"""The `rollout` function generated a list of {len(rollouts)} {type(rollouts[0])}.
After flattening, this list is turned into a {type(transitions)} object containing {len(transitions)} transitions.
The transitions object contains arrays for: {', '.join(transitions.__dict__.keys())}."
"""
)

The `rollout` function generated a list of 56 <class 'imitation.data.types.TrajectoryWithRew'>.
After flattening, this list is turned into a <class 'imitation.data.types.Transitions'> object containing 28000 transitions.
The transitions object contains arrays for: obs, acts, infos, next_obs, dones."



In [24]:
rollouts[0].obs.shape

(501, 4)

In [25]:
obs0 = rollouts[0].obs
plt.plot(obs0[:,3])

After we collected our transitions, it's time to set up our behavior cloning algorithm.

In [26]:
type(env.action_space)

gymnasium.spaces.discrete.Discrete

In [29]:
print(env.action_space.__doc__)

A space consisting of finitely many elements.

    This class represents a finite subset of integers, more specifically a set of the form :math:`\{ a, a+1, \dots, a+n-1 \}`.

    Example:
        >>> from gymnasium.spaces import Discrete
        >>> observation_space = Discrete(2, seed=42) # {0, 1}
        >>> observation_space.sample()
        0
        >>> observation_space = Discrete(3, start=-1, seed=42)  # {-1, 0, 1}
        >>> observation_space.sample()
        -1
    


In [14]:
from imitation.algorithms import bc

bc_trainer = bc.BCO(
    observation_space=env.observation_space,
    action_space=env.action_space,
    state_observations=obvstrans,
    rng=rng,
)

IndexError: tuple index out of range

In [12]:
transitions

Transitions(obs=array([[ 0.02048647,  0.04428037,  0.01656574, -0.03666043],
       [ 0.02137208,  0.2391609 ,  0.01583253, -0.324071  ],
       [ 0.0261553 ,  0.04381713,  0.00935111, -0.0264375 ],
       ...,
       [ 0.05123106,  0.17192517,  0.01526505, -0.16606064],
       [ 0.05466957, -0.02341193,  0.01194384,  0.13139862],
       [ 0.05420133,  0.17153691,  0.01457181, -0.15749238]],
      shape=(28000, 4), dtype=float32), acts=array([1, 0, 1, ..., 0, 1, 0], shape=(28000,)), infos=array([{}, {}, {}, ..., {}, {}, {}], shape=(28000,), dtype=object), next_obs=array([[ 0.02137208,  0.2391609 ,  0.01583253, -0.324071  ],
       [ 0.0261553 ,  0.04381713,  0.00935111, -0.0264375 ],
       [ 0.02703164,  0.23880373,  0.00882236, -0.31615543],
       ...,
       [ 0.05466957, -0.02341193,  0.01194384,  0.13139862],
       [ 0.05420133,  0.17153691,  0.01457181, -0.15749238],
       [ 0.05763207, -0.02379061,  0.01142196,  0.13975175]],
      shape=(28000, 4), dtype=float32), dones=arra

In [13]:
transitions.dones

array([False, False, False, ..., False, False,  True], shape=(28000,))

As you can see the untrained policy only gets poor rewards:

In [14]:
reward_before_training, _ = evaluate_policy(bc_trainer.policy, env, 10)
print(f"Reward before training: {reward_before_training}")

NameError: name 'bc_trainer' is not defined

After training, we can match the rewards of the expert (500):

In [29]:
bc_trainer.train(n_epochs=1)
reward_after_training, _ = evaluate_policy(bc_trainer.policy, env, 10)
print(f"Reward after training: {reward_after_training}")

0batch [00:00, ?batch/s]

---------------------------------
| batch_size        | 32        |
| bc/               |           |
|    batch          | 0         |
|    ent_loss       | -0.000693 |
|    entropy        | 0.693     |
|    epoch          | 0         |
|    l2_loss        | 0         |
|    l2_norm        | 72.5      |
|    loss           | 0.691     |
|    neglogp        | 0.692     |
|    prob_true_act  | 0.5       |
|    samples_so_far | 32        |
---------------------------------


462batch [00:01, 385.56batch/s]

---------------------------------
| batch_size        | 32        |
| bc/               |           |
|    batch          | 500       |
|    ent_loss       | -0.000312 |
|    entropy        | 0.312     |
|    epoch          | 0         |
|    l2_loss        | 0         |
|    l2_norm        | 95.1      |
|    loss           | 0.288     |
|    neglogp        | 0.288     |
|    prob_true_act  | 0.811     |
|    samples_so_far | 16032     |
---------------------------------


862batch [00:02, 386.80batch/s]
875batch [00:02, 356.50batch/s]


Reward after training: 500.0
